## Filter to Pre-Event Latest Earnings Calls

This notebook filters the cleaned transcript dataset to produce the final
text corpus used for NLP analysis. It performs three steps:

1. Keep only earnings calls (remove investor conferences, shareholder meetings, etc.)
2. Extract the true call date from the headline (Capital IQ's `transcript_date`
   reflects when the transcript was uploaded, not when the call occurred — for
   some companies this lag is several weeks)
3. For each company, keep only the most recent earnings call that occurred
   strictly before the DeepSeek R1 release (January 20, 2025)

**Input**: `data/processed/Executives_Transcripts_Cleaned.csv`  
**Output**: `data/processed/Executives_Earnings_Latest_PreEvent.csv`

In [ ]:
import pandas as pd
import re
from pathlib import Path

#project path
project_root = Path.cwd()
while not (project_root / '.git').exists() and project_root.parent != project_root:
    project_root = project_root.parent

INPUT = 'Executives_Transcripts_Cleaned.csv'
OUTPUT =  'Executives_Earnings_Latest_PreEvent.csv'
EVENT_DATE = pd.Timestamp('2025-01-20')


df = pd.read_csv(INPUT)
print(f"Original: {len(df):,} rows, {df['companyid'].nunique()} companies")


# Extract true call date from headline

date_pattern = re.compile(r'([A-Z][a-z]{2}\s+\d{1,2},\s+\d{4})')

def extract_call_date(headline):
    if pd.isna(headline):
        return pd.NaT
    match = date_pattern.search(str(headline))
    if not match:
        return pd.NaT
    return pd.to_datetime(match.group(1), format='%b %d, %Y', errors='coerce')

df['call_date'] = df['headline'].apply(extract_call_date)

#keep only earnings calls based on headline
df = df[df['headline'].str.lower().str.contains('earnings call', na=False)].copy()
print(f"After earnings-call filter: {len(df):,} rows, {df['companyid'].nunique()} companies")

#keep only calls that occurred strictly before the event
df = df[df['call_date'] < EVENT_DATE].copy()
print(f"After pre-event filter:     {len(df):,} rows, {df['companyid'].nunique()} companies")

# keep only the latest call per company
df['latest_call_date'] = df.groupby('companyid')['call_date'].transform('max')
df = df[df['call_date'] == df['latest_call_date']].drop(columns='latest_call_date').copy()
print(f"After latest-call filter:   {len(df):,} rows, {df['companyid'].nunique()} companies")

# sanity check to ensure only one call per company remains
assert df.groupby('companyid')['call_date'].nunique().max() == 1, "Some companies still have multiple calls"
print(f"\nDate range: {df['call_date'].min().date()} to {df['call_date'].max().date()}")
print(f"Records per company — mean: {df.groupby('companyid').size().mean():.1f}, "
      f"median: {df.groupby('companyid').size().median():.0f}, "
      f"max: {df.groupby('companyid').size().max()}")


df.to_csv(OUTPUT, index=False)
print(f"\nSaved to: {OUTPUT}")

Original: 17,316 rows, 404 companies
After earnings-call filter: 9,673 rows, 400 companies
After pre-event filter:     9,673 rows, 400 companies
After latest-call filter:   9,362 rows, 400 companies

Date range: 2024-10-02 to 2025-01-17
Records per company — mean: 23.4, median: 22, max: 177

Saved to: Executives_Earnings_Latest_PreEvent.csv


### Why extract dates from headline instead of using `transcript_date`

Capital IQ's `transcript_date` field is the transcript upload/creation date,
not the actual call date. For example, NVIDIA's Q3 FY2025 earnings call took
place on November 20, 2024, but the transcript was uploaded on February 3,
2025. Using `transcript_date` for event-window filtering would incorrectly
exclude NVIDIA and CrowdStrike — two of the most important companies in our
analysis — even though their calls occurred well before the event.

We extract the true call date by parsing the date string at the end of each
headline (format: `"Company Name, Q3 2024 Earnings Call, Nov 20, 2024"`).